In [1]:
from pathlib import Path
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb

# Ensure Project/src is importable
import sys
PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / 'src').exists():
    sys.path.insert(0, str(PROJECT_DIR))

from src import data_processing as dp

# Config (spec default = 50)
MAX_GAP_FRAMES = 50
RANDOM_STATE = 42

DATA_DIR = PROJECT_DIR.parent / 'PremierLeague_data' / '2024'
DYNAMIC_DIR = DATA_DIR / 'dynamic'
PROCESSED_DIR = DATA_DIR / 'processed'
MODEL_DIR = DATA_DIR / 'pass_prediction_models'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('MODEL_DIR:', MODEL_DIR)

PROJECT_DIR: /home/macaco3001/Projectos/Twelve/twelve-deep-learning/Project
PROCESSED_DIR: /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/processed
MODEL_DIR: /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/pass_prediction_models


In [2]:
# Load events
events = dp.load_premier_league_events(data_dir=DYNAMIC_DIR, limit_matches=None)

# Rescale + normalize so we have x_start_norm/y_start_norm and x_end_norm/y_end_norm consistently
events = dp.rescale_coordinates(events, x_cols=('x_start', 'x_end', 'player_targeted_x_reception'), y_cols=('y_start', 'y_end', 'player_targeted_y_reception'))
events = dp.normalize_attack_direction(events)

# Add regain proxy features on the player-possession action sequence
events = dp.add_opponent_regain_proxy(
    events,
    max_gap_frames=MAX_GAP_FRAMES,
    group_cols=('match_id', 'period'),
    sort_cols=('frame_start', 'frame_end'),
    event_type_col='event_type',
    restrict_event_type='player_possession',
    team_col='team_id',
    frame_start_col='frame_start',
    frame_end_col='frame_end',
    x_start_col='x_start_norm',
    y_start_col='y_start_norm',
    x_end_col='x_end_norm',
    y_end_col='y_end_norm',
)

print('events loaded:', len(events))
print('columns contain regain features:', {'opp_regain_x','opp_regain_gap_frames','dx_to_regain','has_opp_regain_within_gap'} <= set(events.columns))

Found 378 matches in /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/dynamic
  Loaded 50/378 matches...
  Loaded 100/378 matches...
  Loaded 150/378 matches...
  Loaded 200/378 matches...
  Loaded 250/378 matches...
  Loaded 300/378 matches...
  Loaded 350/378 matches...
✅ Loaded 1,811,078 total events from 378 matches
   Event types: 4
   Date range: 1650385 to 2018580
events loaded: 1811078
columns contain regain features: True


In [3]:
# Extract passes (includes offside; success==1 only for pass_outcome=='successful')
passes_raw = dp.extract_pass_events(events)

successful_passes = passes_raw[passes_raw['success'] == 1].copy()
unsuccessful_passes = passes_raw[passes_raw['success'] == 0].copy()

print('passes:', len(passes_raw))
print('  successful:', len(successful_passes))
print('  unsuccessful:', len(unsuccessful_passes))
print('  offside (count):', int((passes_raw.get('pass_outcome') == 'offside').sum()) if 'pass_outcome' in passes_raw.columns else 'n/a')

# Quick sanity checks
assert set(['match_id','period','event_id']).issubset(passes_raw.columns), 'Expected match_id/period/event_id for stable joins'
assert (passes_raw.loc[passes_raw.get('pass_outcome') == 'offside', 'success'] == 0).all() if 'pass_outcome' in passes_raw.columns else True

Extracting pass events from 1,811,078 total events...
  Found 329,716 pass events (end_type='pass')
  Removing 1,003 successful passes with missing reception coordinates
  (0.37% of successful passes)
  Pass outcomes:
    Successful:   272,335 (82.8%)
    Unsuccessful: 55,412 (16.9%)
    Offside:         966 ( 0.3%)
✅ Extracted 328,713 pass events
passes: 328713
  successful: 272335
  unsuccessful: 56378
  offside (count): 966


In [4]:
# Label pass_type for successful passes from reception coordinates
# (We already have *_norm columns; dp.classify_pass_type will compute using reception coords.)
successful_labeled = dp.classify_pass_type(
    successful_passes,
    use_predicted_types=False
)

print('successful_labeled:', len(successful_labeled))
print(successful_labeled['pass_type'].value_counts())

# Target labels
y = successful_labeled['pass_type'].astype(str)

⚠️  Predicted types file not found or use_predicted_types=False
   Falling back to legacy behavior (filter unsuccessful passes)
🔧 Filtered zero-distance passes:
   Removed: 0 (0.00%)
   Remaining: 272,335
   ✓ Derived pass_length and pass_direction from pass_type (ensures consistency with ACTION_NAMES)

✅ Pass type classification complete:
   Pass types: 6 unique
   Expected: 6 types (short/long × backward/lateral/forward)
   Length × Direction = 2 × 3
   Total action space: 6 passes + shoot + carry = 8 actions
successful_labeled: 272335
pass_type
short_lateral     135809
short_forward      55299
short_backward     45270
long_lateral       20024
long_forward       11152
long_backward       4781
Name: count, dtype: int64


In [5]:
# Feature set = prior features + regain-derived features (filled in dp.add_opponent_regain_proxy)
feature_cols = [
    # Spatial/context
    'x_start_norm', 'y_start_norm', 'x_end_norm', 'y_end_norm',
    'third_start', 'channel_start', 'penalty_area_start',
    'game_state', 'team_in_possession_phase_type', 'team_out_of_possession_phase_type',
    'n_teammates_ahead_start', 'n_opponents_ahead_start', 'separation_start',
    'one_touch', 'quick_pass', 'carry', 'forward_momentum', 'is_header', 'hand_pass',
    # Regain-derived
    'opp_regain_gap_frames', 'has_opp_regain_within_gap',
    'dx_to_regain', 'dy_to_regain', 'dist_to_regain', 'angle_to_regain',
]

missing = [c for c in feature_cols if c not in successful_labeled.columns]
if missing:
    raise ValueError(f'Missing feature columns in successful_labeled: {missing}')

X = successful_labeled[feature_cols].copy()

numeric_features = [
    'x_start_norm', 'y_start_norm', 'x_end_norm', 'y_end_norm',
    'n_teammates_ahead_start', 'n_opponents_ahead_start', 'separation_start',
    'opp_regain_gap_frames', 'dx_to_regain', 'dy_to_regain', 'dist_to_regain', 'angle_to_regain',
]
categorical_features = [
    'third_start', 'channel_start', 'game_state',
    'team_in_possession_phase_type', 'team_out_of_possession_phase_type',
]
boolean_features = [
    'penalty_area_start', 'one_touch', 'quick_pass', 'carry', 'forward_momentum', 'is_header', 'hand_pass',
    'has_opp_regain_within_gap',
]

assert set(numeric_features + categorical_features + boolean_features) == set(feature_cols)

# Preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('bool', 'passthrough', boolean_features),
    ],
    remainder='drop'
)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    max_depth=8,
    learning_rate=0.05,
    n_estimators=250,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])

print('Training XGBoost...')
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_val)
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))
print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y_val, y_pred))

Training XGBoost...
                precision    recall  f1-score   support

 long_backward       0.43      0.13      0.21       956
  long_forward       0.87      0.39      0.54      2230
  long_lateral       0.48      0.03      0.05      4005
short_backward       0.53      0.31      0.39      9054
 short_forward       0.53      0.22      0.31     11060
 short_lateral       0.56      0.88      0.68     27162

      accuracy                           0.56     54467
     macro avg       0.57      0.33      0.36     54467
  weighted avg       0.55      0.56      0.50     54467

Confusion matrix (rows=true, cols=pred):
[[  129     0     9   218    25   575]
 [    1   871     7    19   299  1033]
 [   34    22   101   194   120  3534]
 [   69     3    16  2808   251  5907]
 [   18    83    14   431  2464  8050]
 [   48    27    63  1659  1463 23902]]


In [6]:
# Predict pass types for unsuccessful passes
X_unsucc = unsuccessful_passes[feature_cols].copy()
y_unsucc_pred_encoded = pipe.predict(X_unsucc)
y_unsucc_pred = label_encoder.inverse_transform(y_unsucc_pred_encoded)

unsuccessful_with_pred = unsuccessful_passes.copy()
unsuccessful_with_pred['predicted_pass_type'] = y_unsucc_pred

print('Predicted distribution (unsuccessful, before override):')
print(pd.Series(y_unsucc_pred).value_counts(normalize=True).sort_index().mul(100).round(1).astype(str) + '%')

Predicted distribution (unsuccessful, before override):
long_backward      0.0%
long_forward      19.4%
long_lateral       4.4%
short_backward     3.8%
short_forward     32.7%
short_lateral     39.7%
Name: proportion, dtype: object


In [7]:
# Apply direction-only override using regain proxy; keep predicted length
unsuccessful_overridden = dp.override_predicted_pass_direction_only(
    unsuccessful_with_pred,
    predicted_type_col='predicted_pass_type',
    output_col='predicted_pass_type_dir_override',
    max_gap_frames=MAX_GAP_FRAMES,
    success_col='success',
    team_switch_col='team_switch_next',
    gap_col='opp_regain_gap_frames',
    x_end_col='x_end_norm',
    y_end_col='y_end_norm',
    opp_x_col='opp_regain_x',
    opp_y_col='opp_regain_y',
)

n_over = int(unsuccessful_overridden['direction_override_applied'].sum())
pct_over = float(unsuccessful_overridden['direction_override_applied'].mean() * 100)
print(f'Override applied: {n_over:,} ({pct_over:.2f}%)')

print('Predicted distribution (unsuccessful, after override):')
print(unsuccessful_overridden['predicted_pass_type_dir_override'].value_counts(normalize=True).sort_index().mul(100).round(1).astype(str) + '%')

Override applied: 38,641 (68.54%)
Predicted distribution (unsuccessful, after override):
predicted_pass_type_dir_override
long_backward      0.1%
long_forward      19.4%
long_lateral       4.4%
short_backward     5.1%
short_forward     33.4%
short_lateral     37.7%
Name: proportion, dtype: object


In [8]:
# Build full dataset and save outputs (schema of pass_type remains 6-class)
successful_out = successful_labeled.copy()
successful_out['pass_type_source'] = 'actual'

unsuccessful_out = unsuccessful_overridden.copy()
unsuccessful_out['pass_type'] = unsuccessful_out['predicted_pass_type_dir_override']
unsuccessful_out['pass_type_source'] = np.where(
    unsuccessful_out['direction_override_applied'],
    'predicted_dir_override',
    'predicted',
)

all_passes_complete = pd.concat([successful_out, unsuccessful_out], ignore_index=True)

# Save datasets
complete_path = PROCESSED_DIR / 'passes_with_types_complete.parquet'
unsucc_path = PROCESSED_DIR / 'unsuccessful_passes_predicted.parquet'

all_passes_complete.to_parquet(complete_path, index=False)
unsuccessful_out.to_parquet(unsucc_path, index=False)

# Save model artifacts
model_path = MODEL_DIR / 'xgb_pass_type_model_tuned.pkl'
encoder_path = MODEL_DIR / 'label_encoder.pkl'
config_path = MODEL_DIR / 'model_config.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(pipe, f)
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)

config = {
    'max_gap_frames': MAX_GAP_FRAMES,
    'feature_cols': feature_cols,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'boolean_features': boolean_features,
    'classes': list(label_encoder.classes_),
}
with open(config_path, 'wb') as f:
    pickle.dump(config, f)

print('✅ Saved:')
print('  -', complete_path)
print('  -', unsucc_path)
print('  -', model_path)
print('  -', encoder_path)
print('  -', config_path)
print('Direction override applied (unsuccessful):', int(unsuccessful_out['direction_override_applied'].sum()))

✅ Saved:
  - /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/processed/passes_with_types_complete.parquet
  - /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/processed/unsuccessful_passes_predicted.parquet
  - /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/pass_prediction_models/xgb_pass_type_model_tuned.pkl
  - /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/pass_prediction_models/label_encoder.pkl
  - /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/pass_prediction_models/model_config.pkl
Direction override applied (unsuccessful): 38641
